# Lock Manager

Use this notebook to inspect, create, and forcefully remove file locks from the
collaboration SQLite database — without needing to open or close notebooks in the UI.

**Lock key format:** `notebook:<relative-path>`  
**Example:** `notebook:Personal/my_notebook.ipynb`

**Note:** Locking only applies to files inside `Projects/` or `Personal/` folders.
Files outside those folders are never locked regardless of what is in the DB.

## Configuration

Set `DB_PATH` to match your `jupyter_server_ydoc` configuration.
The default path is `/srv/collaboration/collaboration_locks.db`.

Set `NOTEBOOK_PATH` to match your test notebook. 

For the right test, consider the following:

- Run `jupyter lab` from the root directory of the repo.
- Make a `Personal` folder in the root directory.
- Put your test notebook `my_notebook.ipynb` in the `Personal` directory.
- Use this notebook to manually lock and unlock that notebook to test the file locking mechanism.

In [3]:
import sqlite3
import time
from datetime import datetime
from pathlib import Path

# ── EDIT THIS ────────────────────────────────────────────────────────────────
DB_PATH = "/srv/collaboration/collaboration_locks.db"
NOTEBOOK_PATH = "Personal/my_notebook.ipynb"
# Seconds after which a lock without a heartbeat is considered stale.
# Must match the server's lock_ttl_seconds config (default: 30).
TTL_SECONDS = 30
# ─────────────────────────────────────────────────────────────────────────────


def _connect() -> sqlite3.Connection:
    """Open a connection to the lock database."""
    path = Path(DB_PATH)
    if not path.exists():
        raise FileNotFoundError(
            f"Lock DB not found at {DB_PATH}.\n"
            "Check that the server has started at least once, or update DB_PATH above."
        )
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    con.execute("PRAGMA journal_mode=WAL;")
    con.execute(f"PRAGMA busy_timeout=5000;")
    return con


def lock_key(file_path: str, file_type: str = "notebook") -> str:
    """Build the lock key for a given file path and type.

    Parameters
    ----------
    file_path:
        Path relative to the Jupyter root directory.
        Must start with 'Projects/' or 'Personal/' for locking to be enforced.
    file_type:
        Document type as used by the server (almost always 'notebook' for .ipynb files).
    """
    return f"{file_type}:{file_path}"


print(f"DB_PATH = {DB_PATH}")
print(f"NOTEBOOK_PATH = {NOTEBOOK_PATH}")
print(f"TTL     = {TTL_SECONDS}s")
print()
print("Example lock key:", lock_key(NOTEBOOK_PATH))

DB_PATH = /srv/collaboration/collaboration_locks.db
NOTEBOOK_PATH = Personal/my_notebook.ipynb
TTL     = 30s

Example lock key: notebook:Personal/my_notebook.ipynb


## List All Locks

Shows every row in the database, including expired ones (marked with `EXPIRED`).

In [6]:
def list_locks() -> None:
    """Print all locks currently in the database."""
    con = _connect()
    try:
        rows = con.execute(
            "SELECT * FROM doc_locks ORDER BY acquired_at"
        ).fetchall()
    finally:
        con.close()

    if not rows:
        print("Database is empty — no locks.")
        return

    now = time.time()
    print(f"{'Status':<9} {'Lock Key':<55} {'Owner':<20} {'Age (s)':>8} {'Conns':>6}")
    print("-" * 105)
    for row in rows:
        age = now - row["acquired_at"]
        hb_age = now - row["heartbeat_at"]
        expired = hb_age > TTL_SECONDS
        status = "EXPIRED" if expired else "active"
        print(
            f"{status:<9} {row['lock_key']:<55} {row['owner']:<20} {age:>8.1f} {row['connections']:>6}"
        )


list_locks()

Status    Lock Key                                                Owner                 Age (s)  Conns
---------------------------------------------------------------------------------------------------------
active    notebook:Personal/Untitled.ipynb                        77d3e81e19444af8ab0edcbbd7bcf54e     72.0      1
active    notebook:Personal/my_notebook.ipynb                     testuser                  3.9      1


## Lock a File

Inserts a lock row directly into the database, simulating a user opening a notebook.
This causes the next user who opens the file to see the **Read-Only Mode** warning dialog.

**Note:** The lock will expire naturally after `TTL_SECONDS` seconds unless a heartbeat
keeps it alive. To keep it alive, re-run `refresh_lock()` periodically, or use the
`maintain_lock()` helper in the next section.

In [5]:
def lock_file(
    file_path: str,
    owner: str,
    file_type: str = "notebook",
    force: bool = False,
) -> bool:
    """Insert a lock into the database for the given file path.

    Parameters
    ----------
    file_path:
        Relative path from the Jupyter root, e.g. 'Projects/my_notebook.ipynb'.
    owner:
        Username that will appear in the read-only warning dialog.
    file_type:
        Usually 'notebook'.
    force:
        If True, replaces any existing lock (even non-expired ones).
        If False and a non-expired lock already exists, the operation is a no-op.

    Returns
    -------
    True if the lock was created/replaced, False if it was skipped.
    """
    key = lock_key(file_path, file_type)
    now = time.time()

    con = _connect()
    try:
        con.execute("BEGIN IMMEDIATE;")
        row = con.execute(
            "SELECT owner, heartbeat_at FROM doc_locks WHERE lock_key=?", (key,)
        ).fetchone()

        if row is not None:
            hb_age = now - row["heartbeat_at"]
            if hb_age <= TTL_SECONDS and not force:
                print(
                    f"Lock already held by '{row['owner']}' (heartbeat {hb_age:.1f}s ago). "
                    "Use force=True to replace it."
                )
                con.commit()
                return False
            # Replace expired or forced.
            con.execute("DELETE FROM doc_locks WHERE lock_key=?", (key,))

        con.execute(
            "INSERT INTO doc_locks(lock_key, owner, acquired_at, heartbeat_at, connections) "
            "VALUES(?,?,?,?,1)",
            (key, owner, now, now),
        )
        con.commit()
        print(f"Locked  '{key}'  as owner='{owner}'")
        return True
    except Exception:
        con.rollback()
        raise
    finally:
        con.close()


# ── EDIT THESE VALUES ─────────────────────────────────────────────────────────
lock_file(
    file_path=NOTEBOOK_PATH,
    owner="testuser",
)

Locked  'notebook:Personal/my_notebook.ipynb'  as owner='testuser'


True

## Refresh a Lock (Heartbeat)

Updates `heartbeat_at` to the current time, resetting the expiry countdown.
Re-run this cell manually (or use `maintain_lock()` below) to keep a lock alive.

In [8]:
def refresh_lock(file_path: str, owner: str, file_type: str = "notebook") -> bool:
    """Update the heartbeat timestamp for an existing lock.

    Returns True if the row was updated, False if no matching lock was found.
    """
    key = lock_key(file_path, file_type)
    now = time.time()
    con = _connect()
    try:
        con.execute("BEGIN IMMEDIATE;")
        res = con.execute(
            "UPDATE doc_locks SET heartbeat_at=? WHERE lock_key=? AND owner=?",
            (now, key, owner),
        )
        con.commit()
        if res.rowcount == 1:
            print(f"Heartbeat refreshed for '{key}' (owner='{owner}')")
            return True
        else:
            print(f"No matching lock found for key='{key}', owner='{owner}'")
            return False
    except Exception:
        con.rollback()
        raise
    finally:
        con.close()


# ── EDIT THESE VALUES ─────────────────────────────────────────────────────────
refresh_lock(
    file_path=NOTEBOOK_PATH,
    owner="testuser",
)

Heartbeat refreshed for 'notebook:Personal/my_notebook.ipynb' (owner='testuser')


True

## Maintain a Lock (Background Heartbeat Loop)

Keeps a lock alive by refreshing the heartbeat every N seconds.
This simulates a user who has a notebook open and is actively editing it.

**Usage:** Run the cell to start the loop, then **interrupt the kernel** (■ button or `Ctrl+C`)
to stop it. The lock will then expire naturally after `TTL_SECONDS` seconds, or you can
unlock it explicitly with `unlock_file()` below.

In [ ]:
import asyncio


async def maintain_lock(
    file_path: str,
    owner: str,
    file_type: str = "notebook",
    interval_seconds: float = 10.0,
) -> None:
    """Continuously refresh a lock until the cell is interrupted.

    Parameters
    ----------
    file_path:
        Relative path from the Jupyter root.
    owner:
        Username shown in the read-only warning.
    interval_seconds:
        How often to send a heartbeat. Should be less than TTL_SECONDS.
    """
    # Ensure the lock exists before starting the loop.
    lock_file(file_path, owner, file_type)

    print(f"Maintaining lock for '{file_path}' as '{owner}' (interval={interval_seconds}s).")
    print("Interrupt the kernel to stop.")
    try:
        while True:
            await asyncio.sleep(interval_seconds)
            ok = refresh_lock(file_path, owner, file_type)
            if not ok:
                print("Lock was lost — stopping heartbeat loop.")
                break
    except (asyncio.CancelledError, KeyboardInterrupt):
        print("\nHeartbeat loop stopped.")


# ── EDIT THESE VALUES ─────────────────────────────────────────────────────────
await maintain_lock(
    file_path=NOTEBOOK_PATH,
    owner="testuser",
    interval_seconds=10.0,
)

## Unlock a File

Removes a lock owned by a specific user.
Use `force_unlock()` to remove a lock regardless of who owns it.

In [ ]:
def unlock_file(file_path: str, owner: str, file_type: str = "notebook") -> bool:
    """Release a lock that is owned by `owner`.

    Decrements the connection counter; the row is deleted when it reaches zero.
    Returns True if the lock was released, False if no matching lock was found.
    """
    key = lock_key(file_path, file_type)
    now = time.time()
    con = _connect()
    try:
        con.execute("BEGIN IMMEDIATE;")
        row = con.execute(
            "SELECT connections FROM doc_locks WHERE lock_key=? AND owner=?",
            (key, owner),
        ).fetchone()

        if row is None:
            print(f"No lock for key='{key}', owner='{owner}' — nothing to release.")
            con.commit()
            return False

        connections = int(row["connections"])
        if connections <= 1:
            con.execute("DELETE FROM doc_locks WHERE lock_key=? AND owner=?", (key, owner))
            print(f"Unlocked '{key}' (owner='{owner}')")
        else:
            con.execute(
                "UPDATE doc_locks SET connections=connections-1, heartbeat_at=? "
                "WHERE lock_key=? AND owner=?",
                (now, key, owner),
            )
            print(
                f"Decremented connection count for '{key}' (owner='{owner}', "
                f"remaining={connections - 1})"
            )

        con.commit()
        return True
    except Exception:
        con.rollback()
        raise
    finally:
        con.close()


def force_unlock(file_path: str, file_type: str = "notebook") -> bool:
    """Remove a lock for the given file regardless of who holds it.

    Returns True if a lock was deleted, False if no lock existed.
    """
    key = lock_key(file_path, file_type)
    con = _connect()
    try:
        con.execute("BEGIN IMMEDIATE;")
        res = con.execute("DELETE FROM doc_locks WHERE lock_key=?", (key,))
        con.commit()
        if res.rowcount > 0:
            print(f"Force-unlocked '{key}'")
            return True
        else:
            print(f"No lock found for '{key}'")
            return False
    except Exception:
        con.rollback()
        raise
    finally:
        con.close()


# Release a lock owned by a specific user:
unlock_file(
    file_path=NOTEBOOK_PATH,
    owner="testuser",
)

# Or forcefully remove any lock on this file:
# force_unlock("Projects/my_notebook.ipynb")

## Purge All Expired Locks

Removes all rows whose heartbeat has not been updated within `TTL_SECONDS`.
The server does this automatically on every DB access, but this cell lets you do it manually.

In [4]:
def purge_expired() -> int:
    """Delete all expired lock rows and return the number deleted."""
    now = time.time()
    con = _connect()
    try:
        con.execute("BEGIN IMMEDIATE;")
        res = con.execute(
            "DELETE FROM doc_locks WHERE (? - heartbeat_at) > ?",
            (now, TTL_SECONDS),
        )
        con.commit()
        print(f"Purged {res.rowcount} expired lock(s).")
        return res.rowcount
    except Exception:
        con.rollback()
        raise
    finally:
        con.close()


purge_expired()

Purged 2 expired lock(s).


2

## Refresh: Show Current Locks

Re-run this cell at any time to see the current state of the lock database.

In [5]:
list_locks()

Database is empty — no locks.
